In [ ]:
import csv
import sqlite3

import pandas as pd
import numpy as np
import sqlite3 as sq
import os
import csv

import matplotlib.pyplot as plt
%matplotlib inline

# Task 0
Data extraction: get the data from 3 tables & combine it into single `.csv` file.
After that read this file using pandas to create Dataframe.
So it will be all joined data in 1 dataframe. Quick check - should be 74818 rows in it.

In [ ]:
db_path = os.path.abspath('G:\projects\scryping\py-restaurant-data-analysis/db.sqlite3')
connection = sqlite3.connect(db_path)
cursor = connection.cursor()
cursor.execute('SELECT '
                    'restaurant_orderitem.id AS orderitem_id, ' 
                     'restaurant_order.id AS order_id, '
                    'restaurant_order.datetime, '
                    ' restaurant_orderitem.quantity, '
                    'restaurant_orderitem.product_id, '
                    'restaurant_product.name AS product_name, '
                    'restaurant_product.price '
                'FROM restaurant_order '
                'LEFT JOIN restaurant_orderitem ON restaurant_order.id = restaurant_orderitem.order_id '
                'LEFT JOIN restaurant_product ON restaurant_orderitem.product_id = restaurant_product.id;')

rows = cursor.fetchall()
column_names = [description[0] for description in cursor.description]
  
csv_to_file = os.path.abspath('G:\projects\scryping\py-restaurant-data-analysis\output.csv') 

with open(csv_to_file, mode="w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(column_names)
    writer.writerows(rows)
cursor.close()
connection.close()

df = pd.read_csv("G:\projects\scryping\py-restaurant-data-analysis\output.csv")
df.shape

# Task 1
Get Top 10 most popular products in restaurant sold by Quantity.
Count how many times each product was sold and create a pie chart with percentage of popularity (by quantity) for top 10 of them.

Example:

![pie chart](../demo/pie.png)

In [ ]:
columns_to_drop = ["order_id", "orderitem_id", "product_id"]
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns])

In [ ]:
df

In [ ]:

if "datetime" in df.columns:
    df["date"] = pd.to_datetime(df["datetime"])
df.dtypes

In [ ]:
if "datetime" in df.columns:
    df.set_index("date", inplace=True)


In [ ]:
df.drop(columns=["datetime"], inplace=True)

In [ ]:
top_10_products = df.groupby("product_name")["quantity"].sum().nlargest(10)
fig, ax = plt.subplots()
ax.pie(top_10_products, labels=top_10_products.index, autopct='%.1f%%')
plt.title("top_10")
plt.show()

# Task 2
Calculate `Item Price` (Product Price * Quantity) for each Order Item in dataframe.
And Make the same Top 10 pie chart, but this time by `Item Price`. So this chart should describe not the most popular products by quantity, but which products (top 10) make the most money for restaurant. It should be also with percentage.

In [ ]:
df["item_price"] = df["quantity"] * df["price"]
df

In [ ]:
total_revenue = df.groupby("product_name")["item_price"].sum().nlargest(10)
total_revenue

In [ ]:
fig, ax = plt.subplots()
ax.pie(total_revenue, labels=total_revenue.index, autopct='%.1f%%')
plt.title("top_10_item_price")
plt.show()

# Task 3
Calculate `Order Hour` based on `Order Datetime`, which will tell about the specific our the order was created (from 0 to 23). Using `Order Hour` create a bar chart, which will tell the total restaurant income based on the hour order was created. So on x-axis - it will be values from 0 to 23 (hours), on y-axis - it will be the total sum of order prices, which were sold on that hour.

Example:

![bar chart](../demo/bar.png)

In [ ]:
df["order_hour"] = df.index.hour
df

In [ ]:
total_restaurant_income = df.groupby("order_hour")["item_price"].sum()
total_restaurant_income


In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(total_restaurant_income.index, total_restaurant_income.values)
plt.xlabel('Hour of Order')
plt.ylabel('Total Sales')
plt.title("Total restaurant income")
plt.xticks(rotation=45)
plt.show()

# Task 4
Make similar bar chart, but right now with `Order Day Of The Week` (from Monday to Sunday), and also analyze total restaurant income by each day of the week.

In [ ]:
df


In [ ]:
df["order_day_of_the_week"] = df.index.weekday
df["order_day_name"] = df.index.strftime('%A')
df

In [ ]:
total_by_the_day = df.groupby("order_day_name")["total_price"].sum()
total_by_the_day

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(total_by_the_day.index, total_by_the_day.values)
plt.xlabel('Day of Order')
plt.ylabel('Total Sales')
plt.title("Total restaurant income by the day")
plt.xticks(rotation=45)
plt.show()